#### Training on MNIST (`train.py`) – Loss, Gradients, and Loop

This notebook documents and tests the training components defined in `src/train.py`.

Goals:

1. Verify `softmax_cross_entropy_loss()` on simple toy data.
2. Run a single training step on a small MNIST batch (forward → loss → backward → update).
3. Run the full `train()` function and inspect loss/accuracy over epochs.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from data import load_mnist
from model import NeuralNet
from train import (
    softmax_cross_entropy_loss,
    compute_accuracy,
    TrainingConfig,
    train,
)

##### 1. Sanity check `softmax_cross_entropy_loss` on toy logits

We first test the loss and its gradient on a small, hand-constructed example
to confirm that:

- The loss is finite and decreases when the model assigns higher probability to correct classes.
- `dlogits` has the same shape as `logits`.
- The underlying softmax probabilities sum to 1 per example.

In [ ]:
logits = np.array([[1.0, 2.0, 3.0],
                   [1.0, 0.0, 0.0]], dtype=np.float32)
y_true = np.array([2, 0], dtype=np.int64)

loss, dlogits = softmax_cross_entropy_loss(logits, y_true)
print("loss:", loss)
print("dlogits:\n", dlogits)
print("dlogits shape:", dlogits.shape)

##### 2. Single training step on a small batch

We now connect `data.py`, `model.py`, and `train.py` for a tiny experiment:

- Take a small batch from `X_train`.
- Forward pass through `NeuralNet`.
- Compute softmax cross-entropy loss and `dlogits`.
- Call `model.backward(dlogits)` and `model.update_params(...)`.
- Observe whether the loss decreases over repeated steps on this batch. 

In [ ]:
X_train, y_train, X_test, y_test = load_mnist()

model = NeuralNet()
X_batch = X_train[:64]
y_batch = y_train[:64]

learning_rate = 1e-2

for step in range(50):
    logits = model.forward(X_batch)
    loss, dlogits = softmax_cross_entropy_loss(logits, y_batch)

    grads = model.backward(dlogits)
    model.update_params(grads, learning_rate=learning_rate)

    if step % 10 == 0:
        print(f"step {step}, loss {loss:.4f}")

##### 3. Full training run over MNIST

After verifying a single-step update works, we run the full `train()` function
from `train.py` to train on the entire training set with mini-batch SGD. [web:428][web:459]

We track:

- training loss per epoch,
- training accuracy,
- test accuracy.

In [ ]:
config = TrainingConfig(
    num_epochs=5,
    batch_size=64,
    learning_rate=1e-2,
    seed=42,
)

trained_model, history = train(config)

##### 4. Plot training loss and accuracy

We visualize how loss and accuracy evolve over epochs to confirm that the model
is learning and generalizing.

In [ ]:
epochs = np.arange(1, len(history.train_loss) + 1)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history.train_loss, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Loss vs Epochs")

plt.subplot(1, 2, 2)
plt.plot(epochs, history.train_accuracy, marker="o", label="train")
plt.plot(epochs, history.test_accuracy, marker="s", label="test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epochs")
plt.legend()

plt.tight_layout()
plt.show()